# 02 - Brand Specification Generator

##  Objective
This notebook demonstrates how to transform a raw, unstructured brand brief into a structured **Brand Specification** using a Large Language Model (LLM).

The Brand Specification is the **foundation** of the entire BRANDORA pipeline. Every subsequent step (name generation, slogan creation, color palette selection, logo generation) depends on this structured data.

---

##  What This Notebook Does

1. **Loads** a sample brand brief from `test_briefs.json`
2. **Connects** to Groq API (using secure environment variables)
3. **Sends** the brief to an LLM with a carefully crafted prompt
4. **Forces** the LLM to output structured JSON (not free-form text)
5. **Validates** the output to ensure it matches the expected schema

---

##  Key Technical Decisions

### Why Groq?
- Free tier with generous rate limits
- Fast inference (Llama 3.3 70B runs in <1 second)
- Supports JSON mode for structured outputs

### Why JSON Mode?
Without JSON mode, the LLM might output:

In [8]:
import os
import json
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

# 1. قراءة المفاتيح السرية من Kaggle Secrets
user_secrets = UserSecretsClient()
openrouter_key = user_secrets.get_secret("OPENROUTER_API_KEY")
inception_key = user_secrets.get_secret("INCEPTION_API_KEY")

# 2. تهيئة عميل OpenRouter (للنماذج الرخيصة)
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_key,
)

# 3. تهيئة عميل Inception Labs (لنموذج Mercury)
inception_client = OpenAI(
    base_url="https://api.inceptionlabs.ai/v1",
    api_key=inception_key,
)

print("✅ تم الاتصال بالمنصتين بنجاح!")
print("📡 OpenRouter: جاهز للنماذج الرخيصة")
print("📡 Inception Labs: جاهز لـ Mercury")

✅ تم الاتصال بالمنصتين بنجاح!
📡 OpenRouter: جاهز للنماذج الرخيصة
📡 Inception Labs: جاهز لـ Mercury


In [9]:
from huggingface_hub import InferenceClient
from kaggle_secrets import UserSecretsClient

# تعريف عميل Hugging Face (لأنه غير معرّف في الخلايا السابقة)
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
hf_client = InferenceClient(token=hf_token)

# قائمة النماذج المراد اختبارها
models_to_check = [
    {
        "name": "Mercury 2.5 (Inception Labs)",
        "client": inception_client,
        "model_id": "mercury-2.5",
        "extra_params": {"reasoning_effort": "low"}
    },
    {
        "name": "LiquidAI/LFM2.5-2.6B (Hugging Face)",
        "client": hf_client,
        "model_id": "LiquidAI/LFM2.5-2.6B",
        "extra_params": {"max_tokens": 10}
    },
    {
        "name": "Qwen/Qwen3.5-9B (Hugging Face)",
        "client": hf_client,
        "model_id": "Qwen/Qwen3.5-9B",
        "extra_params": {"max_tokens": 10}
    }
]

print("🔍 بدء اختبار النماذج...\n")

for model_info in models_to_check:
    print(f"⏳ جاري اختبار: {model_info['name']}")
    try:
        # إرسال طلب بسيط جداً
        if model_info['client'] == inception_client:
            # Inception Labs يحتاج معالجة خاصة
            response = model_info["client"].chat.completions.create(
                model=model_info["model_id"],
                messages=[{"role": "user", "content": "Hello"}],
                max_tokens=10,
                **model_info.get("extra_params", {})
            )
        else:
            # Hugging Face
            response = model_info["client"].chat_completion(
                model=model_info["model_id"],
                messages=[{"role": "user", "content": "Hello"}],
                **model_info.get("extra_params", {})
            )
        
        # استخراج الرد
        content = response.choices[0].message.content
        if content:
            print(f"   ✅ نجح! الرد: '{content.strip()[:50]}'\n")
        else:
            print(f"   ⚠️ تحذير: النموذج رد لكن المحتوى فارغ\n")
        
    except Exception as e:
        print(f"   ❌ فشل! السبب: {str(e)[:200]}\n")

print("✅ انتهى الاختبار!")

🔍 بدء اختبار النماذج...

⏳ جاري اختبار: Mercury 2.5 (Inception Labs)
   ⚠️ تحذير: النموذج رد لكن المحتوى فارغ

⏳ جاري اختبار: LiquidAI/LFM2.5-2.6B (Hugging Face)
   ❌ فشل! السبب: (Request ID: Root=1-6ab16416-51ad3b766bdf11737e1985d2;894bf59e-155a-4b2b-861d-25bc071f4c04)

403 Forbidden: This authentication method does not have sufficient permissions to call Inference Providers 

⏳ جاري اختبار: Qwen/Qwen3.5-9B (Hugging Face)
   ❌ فشل! السبب: (Request ID: Root=1-6ab16416-70a28adf2f77a0b8084fb532;d07340e1-15e4-4286-b83a-340e08a36814)

403 Forbidden: This authentication method does not have sufficient permissions to call Inference Providers 

✅ انتهى الاختبار!
